# Task 2 Demo Notebook

This notebook documents exploratory data analysis for the animal dataset and shows how to run the training and inference scripts from this repository.

## 1. Dataset EDA

Expected image dataset layout:

```text
data/animals/
├── train/
├── valid/
└── test/
```

Each split should contain at least 10 animal classes in separate folders.

In [ ]:
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

dataset_root = Path('data/animals/train')
records = []
for class_dir in sorted(dataset_root.iterdir()):
    if class_dir.is_dir():
        image_files = [path for path in class_dir.iterdir() if path.is_file()]
        records.append({'animal': class_dir.name, 'count': len(image_files)})

eda_df = pd.DataFrame(records)
eda_df

In [ ]:
ax = eda_df.plot.bar(x='animal', y='count', figsize=(12, 4), legend=False, title='Train split class distribution')
ax.set_ylabel('Images')
plt.tight_layout()

In [ ]:
sample_paths = []
for class_dir in sorted(dataset_root.iterdir()):
    if class_dir.is_dir():
        for image_path in class_dir.iterdir():
            if image_path.is_file():
                sample_paths.append(image_path)
                break

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for axis, image_path in zip(axes.flatten(), sample_paths[:10]):
    axis.imshow(Image.open(image_path).convert('RGB'))
    axis.set_title(image_path.parent.name)
    axis.axis('off')
plt.tight_layout()

## 2. Training commands

The notebook keeps training logic inside Python scripts so the project is reproducible from the command line.

In [ ]:
ner_train_command = '''python3 "Task 2/ner/train.py" \
  --train-file data/ner/train.jsonl \
  --valid-file data/ner/valid.jsonl \
  --output-dir artifacts/ner_model'''

image_train_command = '''python3 "Task 2/image_classification/train.py" \
  --data-dir data/animals \
  --output-dir artifacts/image_model'''

print(ner_train_command)
print()
print(image_train_command)

## 3. Pipeline demo

After both models are trained, the full pipeline should return a single boolean value.

In [ ]:
pipeline_command = '''python3 "Task 2/pipeline.py" \
  --text "There is a cow in the picture." \
  --image-path samples/cow.jpg \
  --ner-model-dir artifacts/ner_model \
  --image-model-dir artifacts/image_model'''

print(pipeline_command)

## 4. Edge cases

- plural mentions: `dogs`, `wolves`, `foxes`;
- several animals in one sentence;
- text with no animal mention;
- image containing an unseen class or several animals.

These cases should be validated after training using the CLI scripts above.